In [1]:
import pandas as pd

In [2]:
df = pd.read_json('../data/processed/cleaned_hadiths.json')
print(df.head())

                              reference           book  \
0  Sahih Bukhari, Chapter 1, Hadith  1   Sahih Bukhari   
1  Sahih Bukhari, Chapter 1, Hadith  2   Sahih Bukhari   
2  Sahih Bukhari, Chapter 1, Hadith  3   Sahih Bukhari   
3  Sahih Bukhari, Chapter 1, Hadith  4   Sahih Bukhari   
4  Sahih Bukhari, Chapter 1, Hadith  5   Sahih Bukhari   

                       chapter  \
0  Revelation - كتاب بدء الوحى   
1  Revelation - كتاب بدء الوحى   
2  Revelation - كتاب بدء الوحى   
3  Revelation - كتاب بدء الوحى   
4  Revelation - كتاب بدء الوحى   

                                             text_en  \
0  Narrated 'Umar bin Al-Khattab:                ...   
1  Narrated 'Aisha:                          (the...   
2  Narrated 'Aisha:                       (the mo...   
3  Narrated Jabir bin 'Abdullah Al-Ansari while t...   
4  Narrated Said bin Jubair:                     ...   

                                             text_ar hadith_no  chapter_no  
0  حدثنا الحميدي عبد الله بن الز

In [3]:
texts = []
for index, row in df.iterrows():
    texts.append(row['text_ar'] +' '+ row['text_en'])

In [4]:
print(texts[:5])

['حدثنا الحميدي عبد الله بن الزبير، قال حدثنا سفيان، قال حدثنا يحيى بن سعيد الأنصاري، قال أخبرني محمد بن إبراهيم التيمي، أنه سمع علقمة بن وقاص الليثي، يقول سمعت عمر بن الخطاب  رضى الله عنه  على المنبر قال سمعت رسول الله صلى الله عليه وسلم يقول \u200f"\u200f إنما الأعمال بالنيات، وإنما لكل امرئ ما نوى، فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما هاجر إليه \u200f"\u200f\u200f.\u200f Narrated \'Umar bin Al-Khattab:                          I heard Allah\'s Apostle saying, "The reward of deeds depends upon the      intentions and every person will get the reward according to what he      has intended. So whoever emigrated for worldly benefits or for a woman     to marry, his emigration was for what he emigrated for."', 'حدثنا عبد الله بن يوسف، قال أخبرنا مالك، عن هشام بن عروة، عن أبيه، عن عائشة أم المؤمنين  رضى الله عنها  أن الحارث بن هشام  رضى الله عنه  سأل رسول الله صلى الله عليه وسلم فقال يا رسول الله كيف يأتيك الوحى فقال رسول الله صلى الله عليه وسلم \u200f"\u200f أ

In [5]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

c:\Users\ammar\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ammar\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ammar\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate De

In [6]:
embeddings = model.encode(texts)
print(embeddings.shape)

(33580, 384)


In [14]:
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
import numpy as np
import faiss
dimension = embeddings_normalized.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_normalized)

In [15]:
# Create a mapping from index of embeddings to hadith metadata

index_to_metadata = {}
for idx, row in df.iterrows():
    index_to_metadata[idx] = {
        'hadith_number': row['hadith_no'],
        'book': row['book'],
        'chapter_number': row['chapter_no'],
        'text_ar': row['text_ar'],
        'text_en': row['text_en']
    }

In [16]:
# Save index and metadata mapping to disk
faiss.write_index(index, '../data/processed/faiss_index.bin')
import pickle
with open('../data/processed/index_to_metadata.pkl', 'wb') as f:
    pickle.dump(index_to_metadata, f)

In [23]:
# test retrieval
query = "What did the Prophet say about offering prayers at home instead of mosque?"
query_embedding = model.encode([query], convert_to_numpy=True)
query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)
k = 20
D, I = index.search(query_embedding_normalized, k)
print(f"Top {k} similar hadiths:")
for i in range(k):
    idx = I[0][i]
    metadata = index_to_metadata[idx]
    print(f"Hadith Number: {metadata['hadith_number']}, Book: {metadata['book']}, Chapter: {metadata['chapter_number']}")
    print(f"Arabic Text: {metadata['text_ar']}")
    print(f"English Text: {metadata['text_en']}")
    print(f"Similarity Score: {D[0][i]:.4f}")
    print("-" * 50)

Top 20 similar hadiths:
Hadith Number:  994 , Book: Sahih Bukhari, Chapter: 14
Arabic Text: حدثنا أبو اليمان، قال أخبرنا شعيب، عن الزهري، عن عروة، أن عائشة، أخبرته أن رسول الله صلى الله عليه وسلم كان يصلي إحدى عشرة ركعة، كانت تلك صلاته  تعني بالليل  فيسجد السجدة من ذلك قدر ما يقرأ أحدكم خمسين آية قبل أن يرفع رأسه، ويركع ركعتين قبل صلاة الفجر، ثم يضطجع على شقه الأيمن حتى يأتيه المؤذن للصلاة‏.‏
English Text: Narrated `Aisha:                     Allah's Apostle used to pray eleven rak`at at night and that was his night prayer and each of his prostrations lasted for a period enough for one of you to recite fifty verses before Allah's Apostle raised his head. He also used to pray two rak`at (Sunna) before the (compulsory) Fajr prayer and then lie down on his right side till the Mu'adh-dhin came to him for the prayer.
Similarity Score: 0.6992
--------------------------------------------------
Hadith Number:  1129 , Book: Sahih Bukhari, Chapter: 19
Arabic Text: حدثنا عبد الله بن يوسف، قال أخب